# Week 7 — Backend Architecture & Cloud Deployment
### Cardiovascular Disease Risk Prediction System (CardioML)

This notebook documents the backend design, REST API endpoints, model serving pipeline, and free cloud deployment configuration for the **CardioML** system according to the Darshan University MLDL SOP Project guidelines (Page 2 & Page 4).

#### Week 7 Objectives:
1. **Backend Integration:** Connect Python Flask application with all backend datasets and serialized ML models.
2. **REST API Development:** Implement `/api/predict` and auxiliary endpoints with validation, scaling, and inference.
3. **Inference Pipeline:** End-to-end processing: JSON/Form Request -> Feature Alignment -> Standard Scaling -> Model Inference -> Probability & Factor Output.
4. **Cloud Deployment Setup:** Prepare zero-configuration deployment manifests (`render.yaml`, `Procfile`, `requirements.txt`, `runtime.txt`, `Dockerfile`) for free hosting.

## 1. Backend Data & Model Connection Architecture
The Flask application loads artifacts initialized during the Week 1–5 training pipeline:
- `model.pkl`: Serialized `GradientBoostingClassifier` trained with SOP parameters (300 estimators, 0.05 lr, max_depth 4, min_samples_leaf 3).
- `scaler.pkl`: Serialized `StandardScaler` for the 7 continuous clinical variables.
- `feature_order.pkl`: Strictly ordered list of the 13 feature columns.
- `model_metadata.json`: Runtime metadata containing training stats, hyperparameters, evaluation metrics, and feature importances.
- `cardio_cleaned.csv`: Cleaned dataset (68,573 records) for on-the-fly dataset summaries.

In [1]:
import pickle
import json
import pandas as pd

# Load model artifacts
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('feature_order.pkl', 'rb') as f:
    feature_order = pickle.load(f)
with open('model_metadata.json', 'r') as f:
    metadata = json.load(f)

print("Model Type:", type(model).__name__)
print("Feature Count:", len(feature_order))
print("Model Test Accuracy:", f"{metadata['performance']['accuracy']}%")
print("Model Test ROC-AUC:", f"{metadata['performance']['roc_auc']}%")

## 2. End-to-End Inference Function
Below is the core prediction logic deployed within the Flask backend endpoint (`/api/predict`):

In [2]:
continuous_cols = ['age_years', 'height', 'weight', 'bmi', 'ap_hi', 'ap_lo', 'pulse_pressure']

def predict_cardio_risk(patient_data):
    """
    Processes patient parameters, scales continuous features, and returns prediction with risk explanation.
    """
    # 1. Calculate derived features if not present
    height_m = patient_data['height'] / 100.0
    bmi = round(patient_data['weight'] / (height_m ** 2), 2)
    pulse_pressure = int(patient_data['ap_hi'] - patient_data['ap_lo'])
    
    patient_dict = dict(patient_data)
    patient_dict['bmi'] = bmi
    patient_dict['pulse_pressure'] = pulse_pressure
    
    # 2. Convert to DataFrame and align columns
    input_df = pd.DataFrame([patient_dict])[feature_order]
    
    # 3. Scale continuous columns
    input_df[continuous_cols] = scaler.transform(input_df[continuous_cols])
    
    # 4. Predict class and risk probability
    pred_class = int(model.predict(input_df)[0])
    pred_proba = float(model.predict_proba(input_df)[0][1])
    
    # 5. Extract contributing risk factors
    risk_factors = []
    if patient_data['ap_hi'] >= 140 or patient_data['ap_lo'] >= 90:
        risk_factors.append(f"Hypertension (BP {patient_data['ap_hi']}/{patient_data['ap_lo']} mmHg)")
    elif patient_data['ap_hi'] >= 120:
        risk_factors.append(f"Prehypertension (Systolic BP {patient_data['ap_hi']} mmHg)")
    if bmi >= 30.0:
        risk_factors.append(f"Obesity (BMI {bmi})")
    elif bmi >= 25.0:
        risk_factors.append(f"Overweight (BMI {bmi})")
    if patient_data['cholesterol'] > 1:
        risk_factors.append("Elevated Cholesterol Levels")
    if patient_data['gluc'] > 1:
        risk_factors.append("Elevated Blood Glucose Levels")
    if patient_data['smoke'] == 1:
        risk_factors.append("Tobacco Smoking History")
    if patient_data['active'] == 0:
        risk_factors.append("Sedentary Lifestyle (Lack of Physical Activity)")
        
    return {
        "prediction": pred_class,
        "probability": round(pred_proba * 100, 1),
        "risk_level": "High Risk" if pred_proba >= 0.50 else "Low Risk",
        "bmi": bmi,
        "pulse_pressure": pulse_pressure,
        "risk_factors": risk_factors
    }

# Test with low-risk patient profile
low_risk_patient = {
    'gender': 1, 'age_years': 32, 'height': 168, 'weight': 62,
    'ap_hi': 115, 'ap_lo': 75, 'cholesterol': 1, 'gluc': 1,
    'smoke': 0, 'alco': 0, 'active': 1
}

# Test with high-risk patient profile
high_risk_patient = {
    'gender': 2, 'age_years': 62, 'height': 172, 'weight': 95,
    'ap_hi': 165, 'ap_lo': 105, 'cholesterol': 3, 'gluc': 2,
    'smoke': 1, 'alco': 1, 'active': 0
}

print("Low Risk Case:", predict_cardio_risk(low_risk_patient))
print("High Risk Case:", predict_cardio_risk(high_risk_patient))

## 3. Flask API Route Summary
| Route | Method | Description |
| :--- | :--- | :--- |
| `/` | GET | Renders the CardioML Landing page with SOP metrics and capability cards |
| `/predict` | GET / POST | Renders CVD assessment form & handles form-based prediction |
| `/api/predict` | POST | JSON API endpoint returning prediction, probability, and risk factor insights |
| `/model` | GET | Displays model hyperparameters, evaluation metrics, and feature importance |
| `/insights` | GET | Displays dataset summary, data cleaning funnel, and healthy target ranges |
| `/disclaimer` | GET | Renders academic and clinical disclaimer |
| `/health` | GET | Health check endpoint for cloud uptime monitoring |

## 4. Free Cloud Hosting Deployment Architecture
To satisfy the Week 7 requirement (*"Final Project Deployment to available free hosting site"*), configuration files are provided for seamless deployment:
1. **Render.com (Recommended Free Hosting):**
   - `render.yaml`: Blueprint definition for automated build and deploy from GitHub.
   - `Procfile`: Declares web dyno process command: `web: gunicorn app:app`.
2. **PythonAnywhere (Alternative Free Host):**
   - Direct WSGI file mapping to `app.py`.
3. **Containerized Deployment (Railway / Koyeb / Fly.io):**
   - `Dockerfile`: Multi-stage, lightweight Python 3.11 slim container.
4. **Complete Documentation:**
   - Refer to `DEPLOYMENT_GUIDE.md` for click-by-click instructions.

## Summary of Week 7 Completion
- Backend server integrated with full data pipeline and scikit-learn models.
- Tested JSON and HTML prediction endpoints with clinical factor extraction.
- Complete cloud hosting package created for zero-cost deployment.